# Test pipeline 

Notebook này kiểm thử retrieval theo **pipeline-level** cho `experiments`:

1. Load `data/data_ready` và kiểm tra split train/val/test.
2. **TextCNN** dự đoán `macro_domain` top-k để lọc ứng viên (cùng một bước cho cả hai nhánh).
3. **Xếp hạng:** hoặc **TF-IDF** trên ứng viên đã lọc, hoặc **Siamese LSTM/BiLSTM** (embedding + dot similarity) trên cùng tập ứng viên.
4. So sánh pipeline trên `qa_test` bằng Recall@K / MRR:
   - `TF-IDF + TextCNN` (raw top-3 topic filter → TF-IDF)
   - `Siamese-only` (full corpus; chọn biến thể `bilstm_1l`, `bilstm_2l`, `lstm_uni`)
   - `Siamese + TextCNN` (raw top-3 topic filter → Siamese; cùng biến thể)

Chạy từ repo root hoặc từ thư mục `experiments`. Nếu checkpoint `.pt` chưa nằm trong repo, trỏ các biến artifact tới Kaggle/input hoặc thư mục local chứa artifact đã train.


In [1]:
from __future__ import annotations

import json
import random
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm


def locate_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "model" / "tokenizer.py").is_file() and (p / "experiments").exists():
            return p
    raise FileNotFoundError("Cannot locate repo root containing model/tokenizer.py and experiments")


REPO_ROOT = locate_repo_root()
sys.path.insert(0, str(REPO_ROOT / "experiments"))
from tokenizer_bootstrap import encode_text, encode_with_mask, simple_tokenize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print("REPO_ROOT:", REPO_ROOT)
print("DEVICE:", DEVICE)


REPO_ROOT: C:\Users\hung\Documents\GitHub\vnlegal-rag
DEVICE: cpu


c:\Users\hung\.conda\envs\data_science\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ---- Test configuration ----
TOPIC_TOPK = 3
RETRIEVE_K = 10
EVAL_SAMPLE_N = 200  # set None for full qa_test
#SIAMESE_VARIANT = "bilstm_2l"  # one of: bilstm_2l, bilstm_1l, lstm_uni
SIAMESE_VARIANT = "bilstm_1l"

TFIDF_MAX_FEATURES = 200_000
ENCODE_BATCH_SIZE = 128 if DEVICE.type == "cuda" else 64
PRECOMPUTE_DOC_EMBEDDINGS = True

DATA_READY_CANDIDATES = [
    REPO_ROOT / "data" / "data_ready_k3",
    REPO_ROOT / "data" / "data_ready",
    REPO_ROOT / "data_ready",
    Path("/kaggle/input/datasets/hngphtrn/legals-v3"),
    Path("/kaggle/input/legals-v3"),
    Path("/kaggle/working/vnlegal-rag/data/data_ready"),
    Path("/kaggle/working/data/data_ready"),
]


def first_dir_with_files(candidates: list[Path], required: list[str]) -> Path | None:
    for p in candidates:
        p = p.expanduser().resolve()
        if p.exists() and all((p / name).is_file() for name in required):
            return p
    return None


DATA_READY_DIR = first_dir_with_files(DATA_READY_CANDIDATES, ["qa_train.csv", "qa_val.csv", "qa_test.csv"])
if DATA_READY_DIR is None:
    raise FileNotFoundError(f"Cannot find  data_ready. Tried: {[str(p) for p in DATA_READY_CANDIDATES]}")

print("DATA_READY_DIR:", DATA_READY_DIR)

In [3]:
def read_ready_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t")


qa_train = read_ready_csv(DATA_READY_DIR / "qa_train.csv")
qa_val = read_ready_csv(DATA_READY_DIR / "qa_val.csv")
qa_test = read_ready_csv(DATA_READY_DIR / "qa_test.csv")

# Prefer remapped split corpus files for  because corpus_full.csv is saved before the bottom-k merge.
corpus_split_paths = [DATA_READY_DIR / name for name in ["corpus_train.csv", "corpus_val.csv", "corpus_test.csv"]]
if all(p.is_file() for p in corpus_split_paths):
    corpus_df = pd.concat([read_ready_csv(p) for p in corpus_split_paths], ignore_index=True)
elif (DATA_READY_DIR / "corpus_full.csv").is_file():
    corpus_df = read_ready_csv(DATA_READY_DIR / "corpus_full.csv")
else:
    raise FileNotFoundError("Need corpus_train/val/test.csv or corpus_full.csv in DATA_READY_DIR")

corpus_df = corpus_df.drop_duplicates(subset=["passage_id"]).reset_index(drop=True)

TEXT_COL = "article_content" if "article_content" in corpus_df.columns else "text"
required_qa_cols = {"question", "passage_id", "macro_domain"}
required_corpus_cols = {"passage_id", "macro_domain", TEXT_COL}
missing_qa = required_qa_cols - set(qa_test.columns)
missing_corpus = required_corpus_cols - set(corpus_df.columns)
if missing_qa or missing_corpus:
    raise ValueError(f"Missing columns | qa_test={missing_qa} corpus={missing_corpus}")

label_maps_path = DATA_READY_DIR / "label_maps.json"
if label_maps_path.is_file():
    with open(label_maps_path, encoding="utf-8") as f:
        label_maps = json.load(f)
    LABEL_LIST = label_maps.get("label_list") or sorted(qa_train["macro_domain"].astype(str).unique().tolist())
else:
    label_maps = {}
    LABEL_LIST = sorted(qa_train["macro_domain"].astype(str).unique().tolist())

print("QA sizes:", {"train": len(qa_train), "val": len(qa_val), "test": len(qa_test)})
print("Corpus passages:", len(corpus_df))
print("Labels:", LABEL_LIST)
display(qa_test.head(3))


QA sizes: {'train': 23311, 'val': 2841, 'test': 2991}
Corpus passages: 9715
Labels: ['Finance & Banking', 'Justice & Dispute Resolution', 'Labor & Insurance', 'State Organization & Admin', 'Transportation', 'other']


,qa_id,passage_id,question,answer,question_type,difficulty,macro_domain,question_count,question_len_char,question_len_word,answer_len_char,answer_len_word,doc_name,strat_label
0,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,Luật Điện lực số 28/2004/QH11 áp dụng đối với ...,Luật Điện lực số 28/2004/QH11 áp dụng đối với ...,factual,easy,other,3,91,19,213,46,Luật Điện lực số 28/2004/QH11 của Quốc hội,other||factual
1,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,Khi có sự khác biệt giữa quy định của Luật Điệ...,"Theo Điều 2 của Luật Điện lực số 28/2004/QH11,...",interpretation,medium,other,3,166,38,391,88,Luật Điện lực số 28/2004/QH11 của Quốc hội,other||interpretation
2,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,Một doanh nghiệp nước ngoài hoạt động trong lĩ...,Theo quy định tại Điều 2 của Luật Điện lực số ...,application,hard,other,3,315,66,576,130,Luật Điện lực số 28/2004/QH11 của Quốc hội,other||application


In [4]:
TEXTCNN_ARTIFACT_CANDIDATES = [
    REPO_ROOT / "experiments" / "textcnn_artifacts_legacy",
    REPO_ROOT / "experiments" / "textcnn_artifacts",
    REPO_ROOT / "artifacts" / "textcnn",
    Path("/kaggle/working/textcnn_artifacts_legacy"),
    Path("/kaggle/input/datasets/hngphtrn/textcnn-artifacts-v1-3"),
    Path("/kaggle/input/textcnn-artifacts-v1-3"),
]

SIAMESE_ARTIFACT_CANDIDATES = {
    "bilstm_2l": [
        REPO_ROOT / "experiments" / "siamese_bilstm2L_artifacts",
        REPO_ROOT / "experiments" / "siamese_bilstm_online_v3_artifacts",
        REPO_ROOT / "artifacts" / "siamese_bilstm2L",
        Path("/kaggle/working/siamese_bilstm_online_v3_artifacts"),
        Path("/kaggle/input/siamese-bilstm-online-v3-artifacts"),
    ],
    "bilstm_1l": [
        REPO_ROOT / "experiments" / "siamese_bilstm1L_artifacts",
        REPO_ROOT / "artifacts" / "siamese_bilstm1L",
        Path("/kaggle/working/siamese_bilstm1L_artifacts"),
        Path("/kaggle/input/siamese-bilstm1l-artifacts"),
    ],
    "lstm_uni": [
        REPO_ROOT / "experiments" / "siamese_lstm_uni_artifacts",
        REPO_ROOT / "artifacts" / "siamese_lstm_uni",
        Path("/kaggle/working/siamese_lstm_uni_artifacts"),
        Path("/kaggle/input/siamese-lstm-uni-artifacts"),
    ],
}

TEXTCNN_WEIGHT_NAMES = ["textcnn_best.pt"]
SIAMESE_WEIGHT_NAMES = [
    "siamese_bilstm_online_best.pt",
    "siamese_bilstm2L_best.pt",
    "siamese_bilstm1L_best.pt",
    "siamese_lstm_uni_best.pt",
    "siamese_lstm_traditional_cosine_best.pt",
    "siamese_best.pt",
]


def find_weight_file(artifact_dir: Path | None, preferred_names: list[str]) -> Path | None:
    if artifact_dir is None or not artifact_dir.exists():
        return None
    for name in preferred_names:
        p = artifact_dir / name
        if p.is_file():
            return p
    matches = sorted(artifact_dir.glob("*.pt")) + sorted(artifact_dir.glob("*.pth"))
    return matches[0] if matches else None


def select_artifact_dir(candidates: list[Path], required_files: list[str], weight_names: list[str]) -> tuple[Path | None, Path | None]:
    fallback = None
    for p in candidates:
        p = p.expanduser().resolve()
        if not (p.exists() and all((p / name).is_file() for name in required_files)):
            continue
        if fallback is None:
            fallback = p
        weight = find_weight_file(p, weight_names)
        if weight is not None:
            return p, weight
    return fallback, find_weight_file(fallback, weight_names)


TEXTCNN_ARTIFACT_DIR, TEXTCNN_WEIGHTS = select_artifact_dir(
    TEXTCNN_ARTIFACT_CANDIDATES,
    ["textcnn_meta.json", "tokenizer_vocab.json"],
    TEXTCNN_WEIGHT_NAMES,
)

if SIAMESE_VARIANT not in SIAMESE_ARTIFACT_CANDIDATES:
    raise ValueError(f"Unknown SIAMESE_VARIANT={SIAMESE_VARIANT}")

SIAMESE_ARTIFACT_DIR, SIAMESE_WEIGHTS = select_artifact_dir(
    SIAMESE_ARTIFACT_CANDIDATES[SIAMESE_VARIANT],
    ["siamese_meta.json", "tokenizer_vocab.json"],
    SIAMESE_WEIGHT_NAMES,
)

print("TEXTCNN_ARTIFACT_DIR:", TEXTCNN_ARTIFACT_DIR)
print("TEXTCNN_WEIGHTS:", TEXTCNN_WEIGHTS)
print("SIAMESE_VARIANT:", SIAMESE_VARIANT)
print("SIAMESE_ARTIFACT_DIR:", SIAMESE_ARTIFACT_DIR)
print("SIAMESE_WEIGHTS:", SIAMESE_WEIGHTS)

if TEXTCNN_WEIGHTS is None:
    print("WARNING: TextCNN checkpoint not found. Add textcnn_best.pt to one candidate directory before neural pipeline testing.")
if SIAMESE_WEIGHTS is None:
    print("WARNING: Siamese checkpoint not found. Add a .pt checkpoint to the selected variant directory before neural pipeline testing.")


TEXTCNN_ARTIFACT_DIR: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\textcnn_artifacts
TEXTCNN_WEIGHTS: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\textcnn_artifacts\textcnn_best.pt
SIAMESE_VARIANT: bilstm_1l
SIAMESE_ARTIFACT_DIR: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\siamese_bilstm1L_artifacts
SIAMESE_WEIGHTS: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\siamese_bilstm1L_artifacts\siamese_bilstm_best.pt


In [5]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, num_classes: int, filter_sizes=(3, 4, 5), num_filters=100, dropout=0.5, embed_dropout=0.0, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_dropout = nn.Dropout(embed_dropout)
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, fs) for fs in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        emb = self.embed_dropout(self.embedding(x)).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            feat = F.relu(conv(emb))
            pooled.append(F.max_pool1d(feat, kernel_size=feat.shape[-1]).squeeze(-1))
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))


class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int, num_layers: int, bidirectional: bool, dropout: float, pad_idx: int):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        if mask is None:
            mask = (x != self.pad_idx).float()
        out, _ = self.lstm(self.embedding(x))
        mask = mask.unsqueeze(-1)
        denom = mask.sum(dim=1).clamp_min(1.0)
        pooled = (out * mask).sum(dim=1) / denom
        return F.normalize(pooled, p=2, dim=1)


class SiameseLSTM(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int, num_layers: int, bidirectional: bool, dropout: float, pad_idx: int):
        super().__init__()
        self.encoder = LSTMEncoder(vocab_size, embed_dim, hidden_size, num_layers, bidirectional, dropout, pad_idx)

    def encode(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        return self.encoder(x, mask)

    def forward(self, q_ids: torch.Tensor, d_ids: torch.Tensor, q_mask: torch.Tensor | None = None, d_mask: torch.Tensor | None = None) -> torch.Tensor:
        q = self.encode(q_ids, q_mask)
        d = self.encode(d_ids, d_mask)
        return F.cosine_similarity(q, d)


def unwrap_state_dict(obj):
    if isinstance(obj, dict) and "state_dict" in obj:
        return obj["state_dict"]
    return obj


def add_encoder_prefix_if_needed(state: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    if any(k.startswith("encoder.") for k in state):
        return state
    mapped = {}
    for k, v in state.items():
        if k.startswith(("embedding.", "lstm.")):
            mapped[f"encoder.{k}"] = v
        else:
            mapped[k] = v
    return mapped


def build_textcnn_from_state_dict(state: dict[str, torch.Tensor], num_classes: int) -> TextCNN:
    emb_w = state["embedding.weight"]
    conv_keys = sorted(k for k in state if k.startswith("convs.") and k.endswith(".weight"))
    filter_sizes = [int(state[k].shape[-1]) for k in conv_keys]
    num_filters = int(state[conv_keys[0]].shape[0])
    model = TextCNN(
        vocab_size=int(emb_w.shape[0]),
        embed_dim=int(emb_w.shape[1]),
        num_classes=num_classes,
        filter_sizes=tuple(filter_sizes),
        num_filters=num_filters,
        dropout=0.0,
        embed_dropout=0.0,
        pad_idx=0,
    )
    return model


def infer_siamese_config(state: dict[str, torch.Tensor], meta: dict) -> dict:
    emb_key = next(k for k in state if k.endswith("embedding.weight"))
    weight_ih_key = next(k for k in state if k.endswith("lstm.weight_ih_l0"))
    emb = state[emb_key]
    hidden_size = int(state[weight_ih_key].shape[0] // 4)
    bidirectional = any("reverse" in k for k in state)
    layer_ids = []
    for k in state:
        marker = "lstm.weight_ih_l"
        if marker in k:
            suffix = k.split(marker, 1)[1].split("_", 1)[0]
            if suffix.isdigit():
                layer_ids.append(int(suffix))
    return {
        "vocab_size": int(emb.shape[0]),
        "embed_dim": int(emb.shape[1]),
        "hidden_size": hidden_size,
        "num_layers": max(layer_ids) + 1 if layer_ids else int(meta.get("num_layers", 1)),
        "bidirectional": bidirectional,
        "dropout": float(meta.get("dropout", 0.3)),
        "pad_idx": 0,
    }


In [6]:
if TEXTCNN_ARTIFACT_DIR is None or TEXTCNN_WEIGHTS is None:
    raise FileNotFoundError("Missing TextCNN artifacts. Need textcnn_meta.json, tokenizer_vocab.json, and textcnn_best.pt.")
if SIAMESE_ARTIFACT_DIR is None or SIAMESE_WEIGHTS is None:
    raise FileNotFoundError("Missing Siamese artifacts. Need siamese_meta.json, tokenizer_vocab.json, and a .pt checkpoint.")

with open(TEXTCNN_ARTIFACT_DIR / "textcnn_meta.json", encoding="utf-8") as f:
    textcnn_meta = json.load(f)
with open(TEXTCNN_ARTIFACT_DIR / "tokenizer_vocab.json", encoding="utf-8") as f:
    textcnn_vocab = json.load(f)

topic_stoi = textcnn_vocab["stoi"]
CNN_MAX_LEN = int(textcnn_meta.get("max_len", 128))
id2label = {int(k): v for k, v in textcnn_meta.get("id2label", {}).items()}
if not id2label:
    id2label = {i: label for i, label in enumerate(LABEL_LIST)}
num_topics = len(id2label)

state_cnn = unwrap_state_dict(torch.load(TEXTCNN_WEIGHTS, map_location=DEVICE))
topic_model = build_textcnn_from_state_dict(state_cnn, num_classes=num_topics).to(DEVICE)
topic_model.load_state_dict(state_cnn)
topic_model.eval()

with open(SIAMESE_ARTIFACT_DIR / "siamese_meta.json", encoding="utf-8") as f:
    siamese_meta = json.load(f)
with open(SIAMESE_ARTIFACT_DIR / "tokenizer_vocab.json", encoding="utf-8") as f:
    siamese_vocab = json.load(f)

siamese_stoi = siamese_vocab["stoi"]
MAX_Q_LEN = int(siamese_meta.get("max_q_len", 64))
MAX_D_LEN = int(siamese_meta.get("max_d_len", 256))

state_s = add_encoder_prefix_if_needed(unwrap_state_dict(torch.load(SIAMESE_WEIGHTS, map_location=DEVICE)))
siamese_cfg = infer_siamese_config(state_s, siamese_meta)
siamese = SiameseLSTM(**siamese_cfg).to(DEVICE)
siamese.load_state_dict(state_s)
siamese.eval()

print("Loaded TextCNN:", TEXTCNN_WEIGHTS)
print("  CNN_MAX_LEN:", CNN_MAX_LEN, "labels:", id2label)
print("Loaded Siamese:", SIAMESE_WEIGHTS)
print("  config:", siamese_cfg)
print("  max_q_len:", MAX_Q_LEN, "max_d_len:", MAX_D_LEN)


Loaded TextCNN: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\textcnn_artifacts\textcnn_best.pt
  CNN_MAX_LEN: 128 labels: {0: 'Finance & Banking', 1: 'Justice & Dispute Resolution', 2: 'Labor & Insurance', 3: 'State Organization & Admin', 4: 'Transportation', 5: 'other'}
Loaded Siamese: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\siamese_bilstm1L_artifacts\siamese_bilstm_best.pt
  config: {'vocab_size': 6227, 'embed_dim': 200, 'hidden_size': 255, 'num_layers': 1, 'bidirectional': True, 'dropout': 0.3, 'pad_idx': 0}
  max_q_len: 64 max_d_len: 256


In [7]:
corpus_df = corpus_df.copy()
corpus_df["_norm_text"] = corpus_df[TEXT_COL].fillna("").astype(str).map(lambda x: " ".join(simple_tokenize(x)))

tfidf = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    max_features=TFIDF_MAX_FEATURES,
)
tfidf_matrix = normalize(tfidf.fit_transform(corpus_df["_norm_text"]), norm="l2", axis=1)

label_to_indices = defaultdict(list)
for i, label in enumerate(corpus_df["macro_domain"].astype(str).tolist()):
    label_to_indices[label].append(i)


def batch_encode_textcnn(texts: list[str]) -> torch.Tensor:
    ids = [encode_text(t, topic_stoi, CNN_MAX_LEN) for t in texts]
    return torch.tensor(ids, dtype=torch.long, device=DEVICE)


@torch.no_grad()
def predict_topic_topk(question: str, k: int = TOPIC_TOPK) -> tuple[list[int], list[str], list[float]]:
    xb = batch_encode_textcnn([question])
    probs = torch.softmax(topic_model(xb), dim=1).squeeze(0).detach().cpu().numpy()
    top_ids = probs.argsort()[::-1][: min(k, len(probs))].tolist()
    return top_ids, [id2label[i] for i in top_ids], probs[top_ids].tolist()


def candidate_indices_for_labels(labels: list[str]) -> np.ndarray:
    indices = []
    for label in labels:
        indices.extend(label_to_indices.get(str(label), []))
    if not indices:
        return np.arange(len(corpus_df))
    return np.array(sorted(set(indices)), dtype=np.int64)


def encode_siamese_texts(texts: list[str], max_len: int, batch_size: int = ENCODE_BATCH_SIZE) -> np.ndarray:
    all_vecs = []
    for start in tqdm(range(0, len(texts), batch_size), leave=False):
        batch = texts[start : start + batch_size]
        enc = [encode_with_mask(t, siamese_stoi, max_len) for t in batch]
        ids = torch.tensor([x[0] for x in enc], dtype=torch.long, device=DEVICE)
        masks = torch.tensor([x[1] for x in enc], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            vecs = siamese.encode(ids, masks).detach().cpu().numpy()
        all_vecs.append(vecs)
    return np.vstack(all_vecs).astype("float32")


doc_embeddings = None
if PRECOMPUTE_DOC_EMBEDDINGS:
    doc_embeddings = encode_siamese_texts(corpus_df[TEXT_COL].fillna("").astype(str).tolist(), MAX_D_LEN)
    print("Precomputed doc embeddings:", doc_embeddings.shape)
print("TF-IDF matrix:", tfidf_matrix.shape)


Precomputed doc embeddings: (9715, 510)
TF-IDF matrix: (9715, 6033)


In [8]:
def format_results(indices: np.ndarray, scores: np.ndarray, extra: dict | None = None) -> pd.DataFrame:
    out = corpus_df.iloc[indices][["passage_id", "macro_domain", TEXT_COL]].copy()
    out.insert(0, "score", scores)
    if not extra:
        return out.reset_index(drop=True)
    n = len(out)
    for key, value in extra.items():
        if isinstance(value, (str, bytes)):
            out[key] = value
        elif isinstance(value, (list, tuple, np.ndarray, pd.Series)):
            try:
                vlen = len(value)
            except TypeError:
                out[key] = value
                continue
            if vlen != n:
                out[key] = pd.Series([value] * n, index=out.index, dtype=object)
            else:
                out[key] = value
        else:
            out[key] = value
    return out.reset_index(drop=True)


def tfidf_scores(question: str, indices: np.ndarray) -> np.ndarray:
    q = " ".join(simple_tokenize(question))
    qv = normalize(tfidf.transform([q]), norm="l2", axis=1)
    return tfidf_matrix[indices].dot(qv.T).toarray().ravel().astype("float32")


def siamese_scores(question: str, indices: np.ndarray) -> np.ndarray:
    q_emb = encode_siamese_texts([question], MAX_Q_LEN, batch_size=1)[0]
    if doc_embeddings is not None:
        d_emb = doc_embeddings[indices]
    else:
        texts = corpus_df.iloc[indices][TEXT_COL].fillna("").astype(str).tolist()
        d_emb = encode_siamese_texts(texts, MAX_D_LEN)
    return d_emb @ q_emb


def retrieve_tfidf_textcnn(question: str, k: int = RETRIEVE_K, topic_topk: int = TOPIC_TOPK) -> pd.DataFrame:
    _, labels, probs = predict_topic_topk(question, topic_topk)
    indices = candidate_indices_for_labels(labels)
    scores = tfidf_scores(question, indices)
    order = np.argsort(scores)[::-1][:k]
    return format_results(indices[order], scores[order], {"method": "tfidf_textcnn", "topic_labels": labels, "topic_probs": probs})


def retrieve_siamese_only(question: str, k: int = RETRIEVE_K) -> pd.DataFrame:
    indices = np.arange(len(corpus_df), dtype=np.int64)
    scores = siamese_scores(question, indices)
    order = np.argsort(scores)[::-1][:k]
    return format_results(indices[order], scores[order], {"method": "siamese_only"})


def retrieve_siamese_textcnn(question: str, k: int = RETRIEVE_K, topic_topk: int = TOPIC_TOPK) -> pd.DataFrame:
    _, labels, probs = predict_topic_topk(question, topic_topk)
    indices = candidate_indices_for_labels(labels)
    scores = siamese_scores(question, indices)
    order = np.argsort(scores)[::-1][:k]
    return format_results(indices[order], scores[order], {"method": "siamese_textcnn", "topic_labels": labels, "topic_probs": probs})


sample_question = str(qa_test.iloc[0]["question"])
print("Question:", sample_question)
print("Topic top-k:", predict_topic_topk(sample_question, TOPIC_TOPK))
print("TF-IDF + TextCNN (topic filter):")
display(retrieve_tfidf_textcnn(sample_question, RETRIEVE_K).head(RETRIEVE_K))
print("Siamese-only (full corpus):")
display(retrieve_siamese_only(sample_question, RETRIEVE_K).head(RETRIEVE_K))
print("Siamese + TextCNN (raw top-3 topic filter):")
display(retrieve_siamese_textcnn(sample_question, RETRIEVE_K).head(RETRIEVE_K))


Question: Luật Điện lực số 28/2004/QH11 áp dụng đối với những đối tượng nào được quy định tại Điều 2?
Topic top-k: ([5, 2, 3], ['other', 'Labor & Insurance', 'State Organization & Admin'], [0.7742956280708313, 0.06194676458835602, 0.053460847586393356])
TF-IDF + TextCNN (topic filter):


,score,passage_id,macro_domain,article_content,method,topic_labels,topic_probs
0,0.412916,du_thao_luat_kinh_doanh_bao_hiem_sua_oi_du_tha...,Labor & Insurance,Điều 2. Đối tượng áp dụng. 1,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
1,0.367942,luat_tiep_cong_dan_cua_quoc_hoi_so_42_2013_qh1...,State Organization & Admin,Điều 35. Hiệu lực thi hành\n\nLuật này có hiệu...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
2,0.357605,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 2. Đối tượng áp dụng\n\nLuật này áp dụng ...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
3,0.338430,luat_can_cuoc_cong_dan_cua_quoc_hoi_so_59_2014...,other,Điều 2. Đối tượng áp dụng\n\nLuật này áp dụng ...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
4,0.294781,du_thao_luat_cong_nghiep_trong_iem_dieu_2_9f51...,other,Điều 2. Đối tượng áp dụng\n\nLuật này áp dụng ...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
5,0.282964,du_thao_luat_bao_hiem_xa_hoi_dieu_92_88221226,Labor & Insurance,Điều 92. Đối tượng áp dụng\n\nĐối tượng áp dụn...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
6,0.281887,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 1. Phạm vi điều chỉnh\n\nLuật này quy địn...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
7,0.281820,luat_trung_cau_y_dan_cua_quoc_hoi_so_96_2015_q...,State Organization & Admin,Điều 2. Đối tượng áp dụng\n\nLuật này áp dụng ...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
8,0.278651,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 7. Các hành vi bị cấm trong hoạt động điệ...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
9,0.278205,du_thao_luat_san_xuat_san_pham_cong_nghiep_tro...,other,Điều 2. Đối tượng áp dụng\n\nLuật này áp dụng ...,tfidf_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."


Siamese + TextCNN (topic filter):


,score,passage_id,macro_domain,article_content,method,topic_labels,topic_probs
0,0.544595,du_thao_luat_kinh_doanh_bao_hiem_sua_oi_du_tha...,Labor & Insurance,Điều 61. Hình thức giải quyết tranh chấp. 28\n...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
1,0.477919,du_thao_luat_cong_nghiep_trong_iem_dieu_28_e5d...,other,Điều 28. Quản lý Cụm liên kết ngành công nghiệ...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
2,0.457148,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 2. Đối tượng áp dụng\n\nLuật này áp dụng ...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
3,0.421627,du_thao_luat_kinh_doanh_bao_hiem_sua_oi_du_tha...,Labor & Insurance,Điều 28. Thời hạn trả tiền bảo hiểm hoặc bồi t...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
4,0.382587,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 34. Trường hợp miễn trừ giấy phép hoạt độ...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
5,0.359947,du_thao_luat_dan_so_lan_2_dieu_12_a6fd5509,State Organization & Admin,Điều 12. Các hành vi bị nghiêm cấm\n\nNghiêm c...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
6,0.345701,luat_thu_o_cua_quoc_hoi_so_39_2024_qh15_dieu_2...,State Organization & Admin,Điều 28. Bảo vệ môi trường\n\n1. Quản lý và bả...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
7,0.339902,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 5. Hợp tác quốc tế trong hoạt động điện l...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
8,0.320255,luat_ien_luc_so_28_2004_qh11_cua_quoc_hoi_dieu...,other,Điều 16. Tiết kiệm trong sử dụng điện\n\n1. Tổ...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."
9,0.318363,luat_hoa_chat_so_06_2007_qh12_cua_quoc_hoi_die...,other,Điều 28. Bao gói hóa chất\n\n1. Bao gói hóa ch...,siamese_textcnn,"[other, Labor & Insurance, State Organization ...","[0.7742956280708313, 0.06194676458835602, 0.05..."


In [9]:
def evaluate_retriever(
    pipeline_name: str,
    retriever,
    qa_df: pd.DataFrame,
    config: str,
    k: int = RETRIEVE_K,
    sample_n: int | None = EVAL_SAMPLE_N,
) -> dict:
    if sample_n is not None and sample_n < len(qa_df):
        eval_df = qa_df.sample(sample_n, random_state=SEED).reset_index(drop=True)
    else:
        eval_df = qa_df.reset_index(drop=True)

    hits_at = {1: 0, 3: 0, 5: 0, 10: 0}
    reciprocal_ranks = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=pipeline_name):
        ranked = retriever(str(row["question"]), k)
        ranked_ids = ranked["passage_id"].astype(str).tolist()
        gold = str(row["passage_id"])

        rank = None
        for i, pid in enumerate(ranked_ids, start=1):
            if pid == gold:
                rank = i
                break
        for cutoff in hits_at:
            if rank is not None and rank <= cutoff:
                hits_at[cutoff] += 1
        reciprocal_ranks.append(0.0 if rank is None else 1.0 / rank)

    n = len(eval_df)
    return {
        "pipeline": pipeline_name,
        "config": config,
        "n_queries": n,
        "Recall@1": hits_at[1] / n,
        "Recall@3": hits_at[3] / n,
        "Recall@5": hits_at[5] / n,
        "Recall@10": hits_at[10] / n,
        "MRR": float(np.mean(reciprocal_ranks)),
    }


results = [
    evaluate_retriever(
        "TF-IDF + TextCNN",
        retrieve_tfidf_textcnn,
        qa_test,
        config=f"topic_topk={TOPIC_TOPK}",
    ),
    evaluate_retriever(
        "Siamese-only",
        retrieve_siamese_only,
        qa_test,
        config=f"variant={SIAMESE_VARIANT}",
    ),
    evaluate_retriever(
        "Siamese + TextCNN",
        retrieve_siamese_textcnn,
        qa_test,
        config=f"variant={SIAMESE_VARIANT}, topic_topk={TOPIC_TOPK}",
    ),
]

metrics_df = (
    pd.DataFrame(results)
    .sort_values("MRR", ascending=False)
    .reset_index(drop=True)
)
display(metrics_df)


Siamese + TextCNN: 100%|██████████| 200/200 [00:03<00:00, 53.52it/s]


,pipeline,config,n_queries,Recall@1,Recall@3,Recall@5,Recall@10,MRR
0,Siamese + TextCNN,"variant=bilstm_1l, topic_topk=3",200,0.605,0.760,0.815,0.870,0.689093
1,TF-IDF + TextCNN,topic_topk=3,200,0.525,0.705,0.790,0.845,0.630312


In [10]:
# Manual query playground
QUESTION = "Người lao động đóng bảo hiểm xã hội bao lâu thì được hưởng lương hưu?"

topic_ids, topic_labels, topic_probs = predict_topic_topk(QUESTION, TOPIC_TOPK)
print("Topic ids:", topic_ids)
print("Topic labels:", topic_labels)
print("Topic probs:", [round(float(x), 4) for x in topic_probs])

print("TF-IDF + TextCNN")
display(retrieve_tfidf_textcnn(QUESTION, RETRIEVE_K, TOPIC_TOPK))
print("Siamese + TextCNN")
display(retrieve_siamese_textcnn(QUESTION, RETRIEVE_K, TOPIC_TOPK))


Topic ids: [2, 5, 4]
Topic labels: ['Labor & Insurance', 'other', 'Transportation']
Topic probs: [0.9819, 0.0061, 0.0041]
TF-IDF + TextCNN


,score,passage_id,macro_domain,article_content,method,topic_labels,topic_probs
0,0.637830,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 59. Thời điểm hưởng lương hưu\n\n1. Đối v...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
1,0.599326,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 78. Bảo lưu thời gian đóng bảo hiểm xã hộ...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
2,0.597715,du_thao_luat_bao_hiem_xa_hoi_dieu_68_ca08a5ae,Labor & Insurance,Điều 68. Trợ cấp một lần khi nghỉ hưu\n\n1. Ng...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
3,0.597331,du_thao_luat_bao_hiem_xa_hoi_dieu_66_aba64730,Labor & Insurance,Điều 66. Mức lương hưu hằng tháng\n\n1. Mức lư...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
4,0.596983,du_thao_luat_bao_hiem_xa_hoi_dieu_103_45ad8794,Labor & Insurance,Điều 103. Bảo lưu thời gian đóng bảo hiểm xã h...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
5,0.590390,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,"Điều 125. Quy định chi tiết\n\nChính phủ, cơ q...",tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
6,0.588001,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 98. Đối tượng và điều kiện hưởng lương hư...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
7,0.581823,du_thao_luat_bao_hiem_xa_hoi_dieu_136_6cc5f10d,Labor & Insurance,Điều 136. Hiệu lực thi hành\n\n1. Luật này có ...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
8,0.580427,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 58. Trợ cấp một lần khi nghỉ hưu\n\n1. Ng...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
9,0.575230,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 85. Mức đóng và phương thức đóng của ngườ...,tfidf_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."


Siamese + TextCNN


,score,passage_id,macro_domain,article_content,method,topic_labels,topic_probs
0,0.537456,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 59. Thời điểm hưởng lương hưu\n\n1. Đối v...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
1,0.528867,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 70. Hưởng bảo hiểm xã hội một lần\n\n1. Đ...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
2,0.508055,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 102. Hưởng bảo hiểm xã hội một lần\n\n1. ...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
3,0.493984,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_58_2014_q...,Labor & Insurance,Điều 21. Trách nhiệm của người sử dụng lao độn...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
4,0.473455,du_thao_luat_bao_hiem_xa_hoi_dieu_32_8eb59236,Labor & Insurance,Điều 32. Mức đóng và phương thức đóng của ngườ...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
5,0.458795,luat_bao_hiem_xa_hoi_cua_quoc_hoi_so_41_2024_q...,Labor & Insurance,Điều 98. Đối tượng và điều kiện hưởng lương hư...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
6,0.449641,du_thao_luat_bao_hiem_xa_hoi_dieu_66_aba64730,Labor & Insurance,Điều 66. Mức lương hưu hằng tháng\n\n1. Mức lư...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
7,0.447622,du_thao_luat_bao_hiem_xa_hoi_dieu_102_58ea45c7,Labor & Insurance,Điều 102. Bảo hiểm xã hội một lần\n\n1. Người ...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
8,0.439902,du_thao_luat_bao_hiem_xa_hoi_dieu_107_a0c9a0cf,Labor & Insurance,"Điều 107. Giải quyết hưởng lương hưu, bảo hiểm...",siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."
9,0.429169,du_thao_luat_bao_hiem_xa_hoi_dieu_70_858d47dd,Labor & Insurance,Điều 70. Bảo hiểm xã hội một lần\n\n1. Người l...,siamese_textcnn,"[Labor & Insurance, other, Transportation]","[0.9819483757019043, 0.006085579749196768, 0.0..."


In [11]:
# Benchmark: TF-IDF + TextCNN vs Siamese variants (+ TextCNN topic filter); export Markdown

SIAMESE_VARIANTS_TO_BENCH = ["bilstm_1l", "bilstm_2l", "lstm_uni"]


def build_markdown_table(df: pd.DataFrame) -> str:
    cols = ["pipeline", "config", "n_queries", "Recall@1", "Recall@3", "Recall@5", "Recall@10", "MRR"]
    lines = [
        "| Pipeline | Config | n_queries | Recall@1 | Recall@3 | Recall@5 | Recall@10 | MRR |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for _, r in df[cols].iterrows():
        lines.append(
            "| {pipeline} | `{config}` | {n_queries} | {r1:.4f} | {r3:.4f} | {r5:.4f} | {r10:.4f} | {mrr:.4f} |".format(
                pipeline=r["pipeline"],
                config=str(r["config"]),
                n_queries=int(r["n_queries"]),
                r1=float(r["Recall@1"]),
                r3=float(r["Recall@3"]),
                r5=float(r["Recall@5"]),
                r10=float(r["Recall@10"]),
                mrr=float(r["MRR"]),
            )
        )
    return "\n".join(lines)


def load_siamese_variant_bundle(variant: str) -> dict:
    if variant not in SIAMESE_ARTIFACT_CANDIDATES:
        raise ValueError(f"Unknown variant: {variant}")

    artifact_dir, weights = select_artifact_dir(
        SIAMESE_ARTIFACT_CANDIDATES[variant],
        ["siamese_meta.json", "tokenizer_vocab.json"],
        SIAMESE_WEIGHT_NAMES,
    )
    if artifact_dir is None or weights is None:
        raise FileNotFoundError(f"Missing Siamese artifacts/checkpoint for variant={variant}")

    with open(artifact_dir / "siamese_meta.json", encoding="utf-8") as f:
        meta = json.load(f)
    with open(artifact_dir / "tokenizer_vocab.json", encoding="utf-8") as f:
        vocab = json.load(f)

    state = add_encoder_prefix_if_needed(unwrap_state_dict(torch.load(weights, map_location=DEVICE)))
    cfg = infer_siamese_config(state, meta)

    model = SiameseLSTM(**cfg).to(DEVICE)
    model.load_state_dict(state)
    model.eval()

    stoi = vocab["stoi"]
    max_q_len = int(meta.get("max_q_len", 64))
    max_d_len = int(meta.get("max_d_len", 256))

    def encode_texts(texts: list[str], max_len: int, batch_size: int = ENCODE_BATCH_SIZE) -> np.ndarray:
        all_vecs = []
        for start in tqdm(range(0, len(texts), batch_size), leave=False, desc=f"encode-{variant}"):
            batch = texts[start : start + batch_size]
            enc = [encode_with_mask(t, stoi, max_len) for t in batch]
            ids = torch.tensor([x[0] for x in enc], dtype=torch.long, device=DEVICE)
            masks = torch.tensor([x[1] for x in enc], dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                vecs = model.encode(ids, masks).detach().cpu().numpy()
            all_vecs.append(vecs)
        return np.vstack(all_vecs).astype("float32")

    texts = corpus_df[TEXT_COL].fillna("").astype(str).tolist()
    doc_emb = encode_texts(texts, max_d_len) if PRECOMPUTE_DOC_EMBEDDINGS else None

    return {
        "variant": variant,
        "artifact_dir": artifact_dir,
        "weights": weights,
        "model": model,
        "stoi": stoi,
        "max_q_len": max_q_len,
        "max_d_len": max_d_len,
        "doc_emb": doc_emb,
        "encode_texts": encode_texts,
    }


def siamese_scores_with_bundle(question: str, indices: np.ndarray, bundle: dict) -> np.ndarray:
    q_emb = bundle["encode_texts"]([question], bundle["max_q_len"], batch_size=1)[0]
    if bundle["doc_emb"] is not None:
        d_emb = bundle["doc_emb"][indices]
    else:
        texts = corpus_df.iloc[indices][TEXT_COL].fillna("").astype(str).tolist()
        d_emb = bundle["encode_texts"](texts, bundle["max_d_len"])
    return d_emb @ q_emb


def make_siamese_only_retriever(bundle: dict):
    def _retrieve(question: str, k: int = RETRIEVE_K) -> pd.DataFrame:
        indices = np.arange(len(corpus_df), dtype=np.int64)
        scores = siamese_scores_with_bundle(question, indices, bundle)
        order = np.argsort(scores)[::-1][:k]
        return format_results(
            indices[order],
            scores[order],
            {"method": "siamese_only", "variant": bundle["variant"]},
        )

    return _retrieve


def make_siamese_retriever(bundle: dict):
    def _retrieve(question: str, k: int = RETRIEVE_K, topic_topk: int = TOPIC_TOPK) -> pd.DataFrame:
        _, labels, probs = predict_topic_topk(question, topic_topk)
        indices = candidate_indices_for_labels(labels)
        scores = siamese_scores_with_bundle(question, indices, bundle)
        order = np.argsort(scores)[::-1][:k]
        return format_results(
            indices[order],
            scores[order],
            {"method": "siamese_textcnn", "variant": bundle["variant"], "topic_labels": labels, "topic_probs": probs},
        )

    return _retrieve


benchmark_results = []

benchmark_results.append(
    evaluate_retriever("TF-IDF + TextCNN", retrieve_tfidf_textcnn, qa_test, config=f"topic_topk={TOPIC_TOPK}")
)

for variant in SIAMESE_VARIANTS_TO_BENCH:
    bundle = load_siamese_variant_bundle(variant)
    print(f"Loaded variant={variant} | weights={bundle['weights']}")

    siamese_only_retriever = make_siamese_only_retriever(bundle)
    siamese_textcnn_retriever = make_siamese_retriever(bundle)

    benchmark_results.append(
        evaluate_retriever(
            "Siamese-only",
            siamese_only_retriever,
            qa_test,
            config=f"variant={variant}",
        )
    )
    benchmark_results.append(
        evaluate_retriever(
            "Siamese + TextCNN",
            siamese_textcnn_retriever,
            qa_test,
            config=f"variant={variant}, topic_topk={TOPIC_TOPK}",
        )
    )

benchmark_df = pd.DataFrame(benchmark_results).sort_values("MRR", ascending=False).reset_index(drop=True)
display(benchmark_df)

md_path = REPO_ROOT / "experiments" / "KET_QUA_RESULTS_V1_3.md"
md_lines = [
    "# Ket qua pipeline-level ",
    "",
    "So sanh `TF-IDF + TextCNN`, `Siamese-only` (full corpus), va `Siamese + TextCNN` (raw top-3 filter),",
    "voi cac bien the Siamese: `bilstm_1l`, `bilstm_2l`, `lstm_uni`.",
    "",
    f"- RETRIEVE_K: `{RETRIEVE_K}`",
    f"- TOPIC_TOPK: `{TOPIC_TOPK}`",
    f"- EVAL_SAMPLE_N: `{EVAL_SAMPLE_N}`",
    "",
    "## Bang ket qua",
    "",
    build_markdown_table(benchmark_df),
    "",
]
md_path.write_text("\n".join(md_lines), encoding="utf-8")
print("Saved markdown results to:", md_path)


TF-IDF + TextCNN: 100%|██████████| 200/200 [00:01<00:00, 111.80it/s]


Loaded variant=bilstm_1l | weights=C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\siamese_bilstm1L_artifacts\siamese_bilstm_best.pt


Siamese + TextCNN: 100%|██████████| 200/200 [00:03<00:00, 51.24it/s]


Loaded variant=bilstm_2l | weights=C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\siamese_bilstm2L_artifacts\siamese_bilstm_best.pt


Siamese + TextCNN: 100%|██████████| 200/200 [00:05<00:00, 34.20it/s]


Loaded variant=lstm_uni | weights=C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\siamese_lstm_uni_artifacts\siamese_lstm_uni_1l_best.pt


Siamese + TextCNN: 100%|██████████| 200/200 [00:03<00:00, 55.88it/s]


,pipeline,config,n_queries,Recall@1,Recall@3,Recall@5,Recall@10,MRR
0,Siamese + TextCNN,"variant=bilstm_1l, topic_topk=3",200,0.605,0.760,0.815,0.870,0.689093
1,Siamese + TextCNN,"variant=lstm_uni, topic_topk=3",200,0.540,0.745,0.805,0.860,0.658679
2,TF-IDF + TextCNN,topic_topk=3,200,0.525,0.705,0.790,0.845,0.630312
3,Siamese + TextCNN,"variant=bilstm_2l, topic_topk=3",200,0.500,0.695,0.755,0.820,0.607768


Saved markdown results to: C:\Users\hung\Documents\GitHub\vnlegal-rag\experiments\KET_QUA_RESULTS_V1_3.md
